# Network-stress model training

This notebook uses the shared `A2/models/NetworkStressModel.py` implementation.
Training-only percentile thresholds define a one-hour-ahead regional stress proxy.
Random Forest and XGBoost are selected using validation macro-F1, then the selected
model is refitted on training plus validation and evaluated on the test period.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "A2/models/NetworkStressModel.py").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from A2.models.NetworkStressModel import NetworkStressModel, prepare_modelling_data

# Generate with A2/A1_data_pipeline/run_pipeline.py, or set an existing transformed file.
DATA_PATH = PROJECT_ROOT / "A2/data/processed/electricity_transformed.parquet"
MODEL_DIRECTORY = PROJECT_ROOT / "A2/saved_models"
data = pd.read_parquet(DATA_PATH)


## Target and chronological partitions

The nominal split is 60% training, 20% validation and 20% test, assigned by target
timestamp. Current/next-hour imputed rows and nonconsecutive prediction pairs are
excluded. Lagged/rolling inputs can still contain previously interpolated values;
this filtering is not a complete guarantee against interpolation-related leakage.


In [ ]:
train, validation, test, features, thresholds = prepare_modelling_data(data)
display(thresholds.rename(columns={0.75: "75th", 0.90: "90th", 0.97: "97th"}))
display(pd.DataFrame({name: part["stress_target_1h"].value_counts(normalize=True) * 100
                     for name, part in [("train", train), ("validation", validation), ("test", test)]}))
print(f"Selected numeric inputs: {len(features)}")


## Train, compare, evaluate and save

This cell trains the full models and writes the model bundle and matching metadata.


In [ ]:
result = NetworkStressModel(model_dir=MODEL_DIRECTORY).train_eval(data)
display(pd.DataFrame(result["validation_results"]).T)
display(pd.Series(result["test_metrics"]))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
matrix = pd.DataFrame(result["confusion_matrix"],
                      index=result["classes"], columns=result["classes"])
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues")
plt.title(f"Test confusion matrix: {result['selected_model']}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()
print(result["bundle_path"])
print(result["metadata_path"])
